In [ ]:
import seaborn as sns

from fau_colors import register_cmaps
register_cmaps()

sns.set_palette("faculties_light")

# Set presentation context
sns.set_context("notebook", font_scale=1)
sns.set_style("whitegrid")
sns.set_style("ticks")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os 
os.chdir('../')

In [ ]:
# Power plots 
df = pd.DataFrame(columns=['participant', 'trial', 'Work', 'value'])
participants = ['P02','P04','P05','P06']
mass_list = [73.35, 89.36, 87.63,44.91]
for participant,mass in zip(participants, mass_list):
    for trial in [1,2,3,4,5,6]:
        if participant in ['Pü01','P03'] or (participant == 'P06' and trial == 5) or (participant == 'P01' and trial > 2):
            continue
        ik_data = pd.read_csv(f"results/{participant}/gait/addB-ignore_phys/IK/Trial{trial}.mot", sep='\t', skiprows=10)
        id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_HIT_body.sto", sep='\t', skiprows=6)
        id2 = pd.read_csv(f"results/{participant}/gait/ID_results//Trial{trial}_ignore_physics.sto", sep='\t', skiprows=6)
        pow1 = np.zeros(ik_data.shape)
        pow2 = np.zeros(ik_data.shape)
        for i, col_a in enumerate(ik_data.columns):
            ik_data.loc[:,col_a] = np.gradient(ik_data[col_a]) * 100
            if col_a in ['time', 'Time', 'Frame#'] or col_a.startswith('pelvis'):
                continue
            try:
                pow1[:,i] = id1[col_a+'_moment'] * ik_data[col_a] / len(ik_data)
                pow2[:,i] = id2[col_a+'_moment'] * ik_data[col_a] / len(ik_data)  
            except:
                pow1[:,i] = id1[col_a+'_force'] * ik_data[col_a] / len(ik_data) 
                pow2[:,i] = id2[col_a+'_force'] * ik_data[col_a] / len(ik_data)
                print(col_a)
        #df.loc[len(df)] = [participant, trial, 'pos_ih', np.sum(pow1[pow1>0])/mass]
        #df.loc[len(df)] = [participant, trial, 'neg_ih', -np.sum(pow1[pow1<0])/mass]
        #df.loc[len(df)] = [participant, trial, 'pos_addb', np.sum(pow2[pow2>0])/mass]
        #df.loc[len(df)] = [participant, trial, 'neg_addb', -np.sum(pow2[pow2<0])/mass]
        df.loc[len(df)] = [participant, trial, 'Negative', -1+np.sum(pow2[pow2>0])/np.sum(pow1[pow1>0])]
        df.loc[len(df)] = [participant, trial, 'Positive', -1+np.sum(pow2[pow2<0])/np.sum(pow1[pow1<0])]
        #df.loc[len(df)] = [participant, trial, 'total_ih', np.sum(np.abs(pow1))/mass]
        #df.loc[len(df)] = [participant, trial, 'total_addb', np.sum(np.abs(pow2))/mass]
        df.loc[len(df)] = [participant, trial, 'Overall', -1+np.sum(np.abs(pow2))/np.sum(np.abs(pow1))]

df2 = df.copy()
x = 'trial'
hue = 'Work'
sns.boxplot(df2, x=x, y='value', hue=hue,showmeans=True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
# Y grid
# Plot the mean for every column
# Rename the xticks
plt.xticks(labels=["1.3 m/s\nlevel", "0.8 m/s\nlevel", "1.3 m/s\nincline", "0.8 m/s\nincline", "1.3 m/s\ndecline", "0.8 m/s\ndecline"], ticks=[0, 1, 2, 3, 4, 5])
plt.ylabel('Power saving: HIT vs addB-IK')
#plt.xlabel('Trial')
# print the mean for every column x and per x and hue
# Get a list of unique values in the column
for value_ in df2[x].unique():
    for hue_ in df2[hue].unique():
        print(f'{x}={value_}, {hue}={hue_}: {df2[(df2[x] == value_) & (df2[hue] == hue_)].value.mean():.2f}')
for hue_ in df2[hue].unique():
    print(f'{hue}={hue_}: {df2[df2[hue] == hue_].value.mean():.3f}')
for value_ in df2[x].unique():
    print(f'{x}={value_}: {df2[df2[x] == value_].value.mean():.3f}')
# Print the mean for every column
print(f'Overall: {df2.value.mean():.3f}')
# Remove the x title
plt.xlabel('')

# save as pdf
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/power_saving.pdf', bbox_inches='tight')
# Print the mean for every column

In [ ]:
collfunc = lambda x: np.mean(np.abs(x))
pipelines = [['HIT', 'body']]#, ['InsideHumans', 'skin']] #['HIT', 'scan']]
participants = ['P02','P04','P05','P06']
mass_list = [73.35, 89.36, 87.63,44.91]
result_df = pd.DataFrame(columns=['participant', 'trial','pipeline', 'dimension', 'value','value_rel'])
for participant,mass in zip(participants, mass_list):
    for trial in range(1,7):
        if participant == "P03" or (participant == 'P06' and trial == 5) or (participant == 'P01' and trial > 2) or (participant == 'P03' and trial == 2):
            continue
        for pipeline in pipelines:
            name, model = pipeline
            name2 = name+model
            #id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_{name}_{model}.sto", sep='\t', skiprows=6)
            try:
                id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_generic_male_body.sto", sep='\t', skiprows=6)
            except:
                id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_generic_female_body.sto", sep='\t', skiprows=6)
            id2 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_ignore_physics.sto", sep='\t', skiprows=6)
            for dim in ['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force', 'pelvis_tilt_moment', 'pelvis_list_moment', 'pelvis_rotation_moment']:
                result_df.loc[len(result_df)] = [participant, trial, name, dim, collfunc(id1[dim])/mass/9.81, (collfunc(id2[dim])-collfunc(id1[dim]))/collfunc(id2[dim])]
            #result_df.loc[len(result_df)] = [participant, trial, name, 'f_norm', np.mean(np.linalg.norm([id1['pelvis_tx_force'], id1['pelvis_ty_force'], id1['pelvis_tz_force']], axis=0))/mass/9.81, 0]
            #result_df.loc[len(result_df)] = [participant, trial, name, 'm_norm', np.mean(np.linalg.norm([id1['pelvis_tilt_moment'], id1['pelvis_list_moment'], id1['pelvis_rotation_moment']], axis=0))/mass/9.81, 0]
        for dim in ['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force', 'pelvis_tilt_moment', 'pelvis_list_moment', 'pelvis_rotation_moment']:
            result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', dim, collfunc(id2[dim])/mass/9.81, 0]
        #result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', 'f_norm', np.mean(np.linalg.norm([id2[dim] for dim in ['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force']], axis=0))/mass/9.81, 0]
        #result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', 'm_norm', np.mean(np.linalg.norm([id2[dim] for dim in ['pelvis_tilt_moment', 'pelvis_list_moment', 'pelvis_rotation_moment']], axis=0))/mass/9.81, 0]
result_df

In [ ]:
import seaborn as sns
sns.boxplot(x='dimension', y='value', data=result_df, hue='pipeline',showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
# Make the x labels vertical
plt.ylabel('Residual Force / Moment [BW%]')
# out the legend to the top right
plt.legend(loc='upper right')
# rename the x ticks
plt.xticks(labels=["$F_{AP}$", "$F_{vert}$", "$F_{lateral}$", "$M_{tilt}$", "$M_{list}$", "$M_{rotation}$"], ticks=[0, 1, 2, 3, 4, 5])
# Save the plot
plt.savefig('plots/force_moment.pdf', bbox_inches='tight')


In [ ]:
HIT = []
generic = []
no_phys = []
for p in ['P02','P04','P05','P06']:

    hit_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/HIT/IK"
    generic_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/generic/IK"
    no_phys_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-ignore_phys/IK"
    # for every file that ends with .sto in the HIT folder

    for file in os.listdir(hit_path):
        if file.endswith(".csv"):
            hit = pd.read_csv(os.path.join(hit_path, file))
            HIT.append(np.array(hit.iloc[:,1:]).mean())
    for file in os.listdir(generic_path):
        if file.endswith(".csv"):
            gen = pd.read_csv(os.path.join(generic_path, file))
            generic.append(np.array(gen.iloc[:,1:]).mean())
    for file in os.listdir(no_phys_path):
            if file.endswith(".csv"):
                no = pd.read_csv(os.path.join(no_phys_path, file))
                no_phys.append(np.array(no.iloc[:,1:]).mean())
import seaborn as sns
import matplotlib.pyplot as plt
# Create a dataframe


In [ ]:
hit = pd.DataFrame(HIT)
gen = pd.DataFrame(generic)
no = pd.DataFrame(no_phys)
hit['MSK model'] = 'HIT'
gen['MSK model'] = 'addB-Opt'
no['MSK model'] = 'Generic'
df = pd.concat([hit, gen, no])
df = df.melt(id_vars='MSK model')

# Make the figure narrower
plt.figure(figsize=(3,6))
sns.boxplot(hue='MSK model', y='value', data=df,showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
plt.ylabel('Marker RMS [m]')
# Remove the tick in the x axis
plt.xticks([])

# Save the plot
plt.savefig('plots/markerrms.pdf', bbox_inches='tight')

In [ ]:
HIT = {
    "pelvis_tx_force": [],
    "pelvis_ty_force": [],
    "pelvis_tz_force": [],
    "pelvis_tilt_moment": [],
    "pelvis_list_moment": [],
    "pelvis_rotation_moment": []
}
generic = {
    "pelvis_tx_force": [],
    "pelvis_ty_force": [],
    "pelvis_tz_force": [],
    "pelvis_tilt_moment": [],
    "pelvis_list_moment": [],
    "pelvis_rotation_moment": []
}
for p, mass in zip(['P02','P04','P05','P06'],mass_list):

    hit_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/HIT/ID"
    generic_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/generic/ID"
    # for every file that ends with .sto in the HIT folder

    for file in os.listdir(hit_path):
        if file.endswith(".sto"):
            hit = pd.read_csv(os.path.join(hit_path, file), sep='\t', skiprows=10)
            for key in HIT.keys():
                HIT[key].append(np.abs(hit[key]).mean()/mass/9.81)
    for file in os.listdir(generic_path):
        if file.endswith(".sto"):
            gen = pd.read_csv(os.path.join(generic_path, file), sep='\t', skiprows=10)
            for key in generic.keys():
                generic[key].append(np.abs(gen[key]).mean()/mass/9.81)


In [ ]:
hit = pd.DataFrame(HIT)
gen = pd.DataFrame(generic)
gen = gen.iloc[:,3:]
hit = hit.iloc[:,3:]
hit['pipeline'] = 'HIT'
gen['pipeline'] = 'generic'
df = pd.concat([hit, gen])
df = df.melt(id_vars='pipeline')
plt.figure(figsize=(3,6))

sns.boxplot(x='variable', y='value', data=df, hue='pipeline', showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})

plt.ylabel('Residual Force [BW%]')
# out the legend to the top right
plt.legend(loc='upper right')
# rename the x ticks
plt.xticks(labels=["$F_{AP}$", "$F_{vert}$", "$F_{lateral}$"], ticks=[0, 1, 2])
# Save the plot
plt.savefig('plots/force.pdf', bbox_inches='tight')

In [ ]:
hit = pd.DataFrame(HIT)
gen = pd.DataFrame(generic)
gen = gen.iloc[:,:3]
hit = hit.iloc[:,:3]
hit['pipeline'] = 'HIT'
gen['pipeline'] = 'generic'
df = pd.concat([hit, gen])
df = df.melt(id_vars='pipeline')
plt.figure(figsize=(3,6))

sns.boxplot(x='variable', y='value', data=df, hue='pipeline', showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})

plt.ylabel('Residual Moment [BW%]')
# out the legend to the top right
plt.legend(loc='upper right')
# rename the x ticks
plt.xticks(labels=["$M_{tilt}$", "$M_{list}$", "$M_{rotation}$"], ticks=[0, 1, 2])
# Save the plot
plt.savefig('plots/residual_moment.pdf', bbox_inches='tight')

In [ ]:
# Compare all powers: HIT-IK vs addB-IK, HIT-IK vs addB-ID, HIT-ID vs addB-ID
addB_ik = {
    "Positive": [],
    "Negative": [],
    "Overall": [],
    "Trial1": [],
    "Trial2":  [],
    "Trial3": [],
    "Trial4": [],
    "Trial5": [],
    "Trial6": []
}
HIT = {
    "Positive": [],
    "Negative": [],
    "Overall": [],
    "Trial1": [],
    "Trial2":  [],
    "Trial3": [],
    "Trial4": [],
    "Trial5": [],
    "Trial6": []
}
addB_id = {
    "Positive": [],
    "Negative": [],
    "Overall": [],
    "Trial1": [],
    "Trial2":  [],
    "Trial3": [],
    "Trial4": [],
    "Trial5": [],
    "Trial6": []
}
HIT_addB_id = {
    "Positive": [],
    "Negative": [],
    "Overall": [],
    "Trial1": [],
    "Trial2":  [],
    "Trial3": [],
    "Trial4": [],
    "Trial5": [],
    "Trial6": []
}
for p in ['P02','P04','P05','P06']:
    hit_path_id = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/HIT/ID"
    generic_path_id = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/generic/ID"
    hit_path_ik = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/ID_results"
    ik_data_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-ignore_phys/IK"
    for trial in range(1,7):
        if p == "P03" or (p == 'P06' and trial == 5) or (p == 'P01' and trial > 2) or (p == 'P03' and trial == 2):
            continue
        ik_data = pd.read_csv(os.path.join(ik_data_path, f"Trial{trial}.mot"), sep='\t', skiprows=10)
        id1 = pd.read_csv(os.path.join(hit_path_ik, f"Trial{trial}_HIT_body.sto"), sep='\t', skiprows=6)
        id2 = pd.read_csv(os.path.join(hit_path_ik, f"Trial{trial}_ignore_physics.sto"), sep='\t', skiprows=6)
        pow1 = np.zeros(ik_data.shape)
        pow2 = np.zeros(ik_data.shape)
        for i, col_a in enumerate(ik_data.columns):
            ik_data.loc[:,col_a] = np.gradient(ik_data[col_a]) * 100
            if col_a in ['time', 'Time', 'Frame#'] or col_a.startswith('pelvis'):
                continue
            try:
                pow1[:,i] = id1[col_a+'_moment'] * ik_data[col_a] / len(ik_data)
                pow2[:,i] = id2[col_a+'_moment'] * ik_data[col_a] / len(ik_data)  
            except:
                pow1[:,i] = id1[col_a+'_force'] * ik_data[col_a] / len(ik_data) 
                pow2[:,i] = id2[col_a+'_force'] * ik_data[col_a] / len(ik_data)
                print(col_a)
        #df.loc[len(df)] = [participant, trial, 'pos_ih', np.sum(pow1[pow1>0])/mass]
        #df.loc[len(df)] = [participant, trial, 'neg_ih', -np.sum(pow1[pow1<0])/mass]
        #df.loc[len(df)] = [participant, trial, 'pos_addb', np.sum(pow2[pow2>0])/mass]
        #df.loc[len(df)] = [participant, trial, 'neg_addb', -np.sum(pow2[pow2<0])/mass]
        HIT['Positive'].append(np.sum(pow1[pow1>0])/mass)
        HIT['Negative'].append(-np.sum(pow1[pow1<0])/mass)
        HIT['Overall'].append(np.sum(np.abs(pow1))/mass)
        addB_ik['Positive'].append(np.sum(pow2[pow2>0])/mass)
        addB_ik['Negative'].append(-np.sum(pow2[pow2<0])/mass)
        addB_ik['Overall'].append(np.sum(np.abs(pow2))/mass)
        HIT[f"Trial{trial}"].append(np.sum(np.abs(pow1))/mass)
        addB_ik[f"Trial{trial}"].append(np.sum(np.abs(pow2))/mass)

    hit_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/HIT/ID"
    generic_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/generic/ID"
    # for every file that ends with .sto in the HIT folder


In [ ]:
for p in ['P02','P04','P05','P06']:
    for file in os.listdir(hit_path):
        if True:
            if file.endswith(".sto"):
                ik_file = file.replace('.sto', '.mot').replace('ID', 'IK').replace('id','ik')
                ik_path = hit_path.replace('ID', 'IK')
                ik_data = pd.read_csv(os.path.join(ik_path, ik_file), sep='\t', skiprows=10)
                hit = pd.read_csv(os.path.join(hit_path, file), sep='\t', skiprows=10)
                pow = np.zeros(hit.shape)
                for i, col_a in enumerate(ik_data.columns):
                    hit.loc[:,col_a] = np.gradient(ik_data[col_a]) * 100
                    if col_a in ['time', 'Time', 'Frame#'] or col_a.startswith('pelvis'):
                        continue
                    try:
                        pow[:,i] = hit[col_a+'_moment'] * hit[col_a] / len(hit)
                    except:
                        pow[:,i] = hit[col_a+'_force'] * hit[col_a] / len(hit)
                        print(col_a)
                #df.loc[len(df)] = [participant, trial, 'pos_ih', np.sum(pow1[pow1>0])/mass]
                HIT_addB_id['Positive'].append(np.sum(pow[pow>0])/mass)
                HIT_addB_id['Negative'].append(-np.sum(pow[pow<0])/mass)
                HIT_addB_id['Overall'].append(np.sum(np.abs(pow))/mass)
                trial = file.split('rial')[1][0]
                HIT_addB_id[f"Trial{trial}"].append(np.sum(np.abs(pow))/mass)
        #except:
        #    print(file)
    for file in os.listdir(generic_path):
        try:
            if file.endswith(".sto"):
                ik_file = file.replace('.sto', '.mot').replace('ID', 'IK').replace('id','ik')
                ik_path = hit_path.replace('ID', 'IK')
                ik_data = pd.read_csv(os.path.join(ik_path, ik_file), sep='\t', skiprows=10)
                gen = pd.read_csv(os.path.join(generic_path, file), sep='\t', skiprows=10)
                pow = np.zeros(gen.shape)
                for i, col_a in enumerate(ik_data.columns):
                    gen.loc[:,col_a] = np.gradient(ik_data[col_a]) * 100
                    if col_a in ['time', 'Time', 'Frame#'] or col_a.startswith('pelvis'):
                        continue
                    try:
                        pow[:,i] = gen[col_a+'_moment'] * gen[col_a] / len(gen)
                    except:
                        pow[:,i] = gen[col_a+'_force'] * gen[col_a] / len(gen)
                        print(col_a)
                addB_id['Positive'].append(np.sum(pow[pow>0])/mass)
                addB_id['Negative'].append(-np.sum(pow[pow<0])/mass)
                addB_id['Overall'].append(np.sum(np.abs(pow))/mass)
                trial = file.split('rial')[1][0]
                addB_id[f"Trial{trial}"].append(np.sum(np.abs(pow))/mass)

        except:
            print(file)

In [ ]:
addB_ik

In [ ]:
addB_ik_ = pd.DataFrame({key:addB_ik[key] for key in ['Negative','Positive','Overall']})
HIT_ = pd.DataFrame({key:HIT[key] for key in ['Negative','Positive','Overall']})
HIT_addB_id_ =pd.DataFrame({key:HIT_addB_id[key] for key in ['Negative','Positive','Overall']})
addB_id_ = pd.DataFrame({key:addB_id[key] for key in ['Negative','Positive','Overall']})
addB_ik_['Pipeline'] = 'Generic'
HIT_['Pipeline'] = 'HIT'
HIT_addB_id_['Pipeline'] = 'HIT-addB'
addB_id_['Pipeline'] = 'addB-ID'
df = pd.concat([addB_ik_, HIT_])
df = df.melt(id_vars='Pipeline')
plt.figure(figsize=(4,6))

# Make the plot but use multiple colors
sns.boxplot(y='value', data=df, x="variable", hue='Pipeline', showfliers=False, showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
plt.xlabel('')
plt.ylabel('Power [W/kg]')
# save as pdf
plt.savefig('plots/power_comparison.pdf', bbox_inches='tight')

In [ ]:
addB_ik_ = pd.DataFrame({key:addB_ik[key] for key in ['Negative','Positive','Overall']})
HIT_ = pd.DataFrame({key:HIT[key] for key in ['Negative','Positive','Overall']})
HIT_addB_id_ =pd.DataFrame({key:HIT_addB_id[key] for key in ['Negative','Positive','Overall']})
addB_id_ = pd.DataFrame({key:addB_id[key] for key in ['Negative','Positive','Overall']})
addB_ik_['Pipeline'] = 'Generic'
HIT_['Pipeline'] = 'HIT'
HIT_addB_id_['Pipeline'] = 'HIT-addB'
addB_id_['Pipeline'] = 'addB-ID'
df = pd.concat([addB_id_, HIT_addB_id_])
df = df.melt(id_vars='Pipeline')
# Make the plot but use multiple colors
plt.figure(figsize=(4,6))

sns.boxplot(y='value', data=df, x="variable", hue='Pipeline', showfliers=False, showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
plt.xlabel('')
plt.ylabel('Power [W/kg]')
# save as pdf
plt.savefig('plots/power_comparison_id.pdf', bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(10,4))
for t in range(1,7):
    ax = plt.subplot(1,6,t)
    addB_ik_ = pd.DataFrame({key:addB_ik[key] for key in [f"Trial{t}"]})
    HIT_ = pd.DataFrame({key:HIT[key] for key in [f"Trial{t}"]})
    HIT_addB_id_ =pd.DataFrame({key:HIT_addB_id[key] for key in [f"Trial{t}"]})
    addB_id_ = pd.DataFrame({key:addB_id[key] for key in [f"Trial{t}"]})
    addB_ik_['Pipeline'] = 'Generic'
    HIT_['Pipeline'] = 'HIT'
    HIT_addB_id_['Pipeline'] = 'HIT-addB'
    addB_id_['Pipeline'] = 'addB-ID'
    df = pd.concat([addB_id_, HIT_addB_id_, addB_ik_, HIT_])
    df = df.melt(id_vars='Pipeline')
    # Make the plot but use multiple colors

    sns.boxplot(y='value', data=df, x="variable", hue='Pipeline', showfliers=False, showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
    plt.xlabel('')
    if t != 6:
        try:
            ax.get_legend().remove()
        except:
            pass
    plt.ylabel('Power [W/kg]')
plt.tight_layout()
    # save as pdf
plt.savefig('plots/power_comparison_id.pdf', bbox_inches='tight')

In [ ]:
df

In [ ]:
(addB_id_.iloc[:,2].mean()-HIT_addB_id_.iloc[:,2].mean())/addB_id_.iloc[:,2].mean()

In [ ]:
addB_id_.iloc[:,2].mean()

In [ ]:
collfunc = lambda x: np.mean(np.abs(x))
pipelines = [['HIT', 'body']]#, ['InsideHumans', 'skin']] #['HIT', 'scan']]
participants = ['P02','P04','P05','P06']
mass_list = [73.35, 89.36, 87.63,44.91]
result_df = pd.DataFrame(columns=['participant', 'trial','pipeline', 'dimension', 'value','value_rel'])
for participant,mass in zip(participants, mass_list):
    for trial in range(1,7):
        if participant == "P03" or (participant == 'P06' and trial == 5) or (participant == 'P01' and trial > 2) or (participant == 'P03' and trial == 2):
            continue
        for pipeline in pipelines:
            name, model = pipeline
            name2 = name+model
            id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_{name}_{model}.sto", sep='\t', skiprows=6)
            try:
                id3 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_generic_male_body.sto", sep='\t', skiprows=6)
            except:
                id3 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_generic_female_body.sto", sep='\t', skiprows=6)
            id2 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_ignore_physics.sto", sep='\t', skiprows=6)
            for dim in ['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force', 'pelvis_tilt_moment', 'pelvis_list_moment', 'pelvis_rotation_moment']:
                result_df.loc[len(result_df)] = [participant, trial, name, dim, collfunc(id1[dim])/mass/9.81, (collfunc(id2[dim])-collfunc(id1[dim]))/collfunc(id2[dim])]
                result_df.loc[len(result_df)] = [participant, trial, "SMPL-generic", dim, collfunc(id3[dim])/mass/9.81, (collfunc(id2[dim])-collfunc(id1[dim]))/collfunc(id2[dim])]

            #result_df.loc[len(result_df)] = [participant, trial, name, 'f_norm', np.mean(np.linalg.norm([id1['pelvis_tx_force'], id1['pelvis_ty_force'], id1['pelvis_tz_force']], axis=0))/mass/9.81, 0]
            #result_df.loc[len(result_df)] = [participant, trial, name, 'm_norm', np.mean(np.linalg.norm([id1['pelvis_tilt_moment'], id1['pelvis_list_moment'], id1['pelvis_rotation_moment']], axis=0))/mass/9.81, 0]
        for dim in ['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force', 'pelvis_tilt_moment', 'pelvis_list_moment', 'pelvis_rotation_moment']:
            result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', dim, collfunc(id2[dim])/mass/9.81, 0]
        #result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', 'f_norm', np.mean(np.linalg.norm([id2[dim] for dim in ['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force']], axis=0))/mass/9.81, 0]
        #result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', 'm_norm', np.mean(np.linalg.norm([id2[dim] for dim in ['pelvis_tilt_moment', 'pelvis_list_moment', 'pelvis_rotation_moment']], axis=0))/mass/9.81, 0]
result_df

In [ ]:
import seaborn as sns
plt.figure(figsize=(5,6))

sns.boxplot(x='dimension', y='value', data=result_df, hue='pipeline',showmeans = True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
# Make the x labels vertical
plt.ylabel('Residual Force / Moment [BW%]')
# out the legend to the top right
plt.legend(loc='upper right')
# rename the x ticks
plt.xticks(labels=["$F_{AP}$", "$F_{vert}$", "$F_{lateral}$", "$M_{tilt}$", "$M_{list}$", "$M_{rotation}$"], ticks=[0, 1, 2, 3, 4, 5])
# Save the plot
plt.savefig('plots/force_moment_w_smpl.pdf', bbox_inches='tight')

In [ ]:
# Power plots 
df = pd.DataFrame(columns=['participant', 'trial', 'Work', 'value','Model'])
participants = ['P02','P04','P05','P06']
mass_list = [73.35, 89.36, 87.63,44.91]
for participant,mass in zip(participants, mass_list):
    for trial in [1,2,3,4,5,6]:
        if participant in ['Pü01','P03'] or (participant == 'P06' and trial == 5) or (participant == 'P01' and trial > 2):
            continue
        ik_data = pd.read_csv(f"results/{participant}/gait/addB-ignore_phys/IK/Trial{trial}.mot", sep='\t', skiprows=10)
        id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_HIT_body.sto", sep='\t', skiprows=6)
        try:
            id3 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_generic_male_body.sto", sep='\t', skiprows=6)
        except:
            id3 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}_generic_female_body.sto", sep='\t', skiprows=6)
        id2 = pd.read_csv(f"results/{participant}/gait/ID_results//Trial{trial}_ignore_physics.sto", sep='\t', skiprows=6)
        pow1 = np.zeros(ik_data.shape)
        pow2 = np.zeros(ik_data.shape)#
        pow3 = np.zeros(ik_data.shape)
        for i, col_a in enumerate(ik_data.columns):
            ik_data.loc[:,col_a] = np.gradient(ik_data[col_a]) * 100
            if col_a in ['time', 'Time', 'Frame#'] or col_a.startswith('pelvis'):
                continue
            try:
                pow1[:,i] = id1[col_a+'_moment'] * ik_data[col_a] / len(ik_data)
                pow2[:,i] = id2[col_a+'_moment'] * ik_data[col_a] / len(ik_data)  
                pow3[:,i] = id3[col_a+'_moment'] * ik_data[col_a] / len(ik_data)
            except:
                pow1[:,i] = id1[col_a+'_force'] * ik_data[col_a] / len(ik_data) 
                pow2[:,i] = id2[col_a+'_force'] * ik_data[col_a] / len(ik_data)
                pow3[:,i] = id3[col_a+'_force'] * ik_data[col_a] / len(ik_data)
                print(col_a)
        #df.loc[len(df)] = [participant, trial, 'pos_ih', np.sum(pow1[pow1>0])/mass]
        #df.loc[len(df)] = [participant, trial, 'neg_ih', -np.sum(pow1[pow1<0])/mass]
        #df.loc[len(df)] = [participant, trial, 'pos_addb', np.sum(pow2[pow2>0])/mass]
        #df.loc[len(df)] = [participant, trial, 'neg_addb', -np.sum(pow2[pow2<0])/mass]
        df.loc[len(df)] = [participant, trial, 'Negative', -1+np.sum(pow2[pow2>0])/np.sum(pow1[pow1>0]),'HIT']
        df.loc[len(df)] = [participant, trial, 'Positive', -1+np.sum(pow2[pow2<0])/np.sum(pow1[pow1<0]),'HIT']
        #df.loc[len(df)] = [participant, trial, 'total_ih', np.sum(np.abs(pow1))/mass]
        #df.loc[len(df)] = [participant, trial, 'total_addb', np.sum(np.abs(pow2))/mass]
        df.loc[len(df)] = [participant, trial, 'Overall', -1+np.sum(np.abs(pow2))/np.sum(np.abs(pow1)),'HIT']
        df.loc[len(df)] = [participant, trial, 'Negative', -1+np.sum(pow3[pow3>0])/np.sum(pow1[pow1>0]),'SMPL']
        df.loc[len(df)] = [participant, trial, 'Positive', -1+np.sum(pow3[pow3<0])/np.sum(pow1[pow1<0]),'SMPL']
        #df.loc[len(df)] = [participant, trial, 'total_ih', np.sum(np.abs(pow1))/mass]
        #df.loc[len(df)] = [participant, trial, 'total_addb', np.sum(np.abs(pow2))/mass]
        df.loc[len(df)] = [participant, trial, 'Overall', -1+np.sum(np.abs(pow3))/np.sum(np.abs(pow1)),'SMPL']

df2 = df.copy()
x = 'trial'
hue = 'Model'
plt.figure(figsize=(5,6))
sns.boxplot(df2, x=x, y='value', hue=hue,showmeans=True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'8'})
# Y grid
# Plot the mean for every column
# Rename the xticks
plt.xticks(labels=["1.3 m/s\nlevel", "0.8 m/s\nlevel", "1.3 m/s\nincline", "0.8 m/s\nincline", "1.3 m/s\ndecline", "0.8 m/s\ndecline"], ticks=[0, 1, 2, 3, 4, 5])
plt.ylabel('Power saving: HIT vs addB-IK')
#plt.xlabel('Trial')
# print the mean for every column x and per x and hue
# Get a list of unique values in the column
for value_ in df2[x].unique():
    for hue_ in df2[hue].unique():
        print(f'{x}={value_}, {hue}={hue_}: {df2[(df2[x] == value_) & (df2[hue] == hue_)].value.mean():.2f}')
for hue_ in df2[hue].unique():
    print(f'{hue}={hue_}: {df2[df2[hue] == hue_].value.mean():.3f}')
for value_ in df2[x].unique():
    print(f'{x}={value_}: {df2[df2[x] == value_].value.mean():.3f}')
# Print the mean for every column
print(f'Overall: {df2.value.mean():.3f}')
# Remove the x title
plt.xlabel('')

# save as pdf
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/power_saving_w_smpl.pdf', bbox_inches='tight')
# Print the mean for every column

In [ ]:
p = 'P04'
trial = 3
mass = 44.91

grf_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/data/{p}/gait/Trial{trial}_filtered.mot"
grf = pd.read_csv(grf_path, sep='\t', skiprows=5)

grf_y = grf['ground_force_vy'].to_numpy().copy()
grf_y[grf_y<30] = 0
grf_y[grf_y>0] = 1

# Find all rising edges
rising_edges = np.where(np.diff(grf_y) == 1)[0]
# Plot the data from the first rising edge to the next falling edge

grf_y = grf['ground_force_vy'].to_numpy()
gc_list = []    
for i in range(len(rising_edges)-1):
    start = rising_edges[i]
    end = rising_edges[i+1]
    gc_list.append((start, end))

# plot the lengths of each gaait cycle
plt.figure(figsize=(5,6))
mean_gc_len = np.mean([end-start for start, end in gc_list])
std_gc_len = np.std([end-start for start, end in gc_list])
gc_filtered = []
for start, end in gc_list:
    if abs(end-start-mean_gc_len) < std_gc_len:
        gc_filtered.append((start, end))

# loadID 
id1 = pd.read_csv(f"results/{p}/gait/ID_results/Trial{trial}_HIT_body.sto", sep='\t', skiprows=6)
dim = "hip_flexion_r_moment"
# Grab all gait cycles
gc_list = []
for gc in gc_filtered:
    start, end = gc
    # Get the index of the first and last gait cycle
    id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
    # interpolate to 100 points
    id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
    gc_list.append(id1_gc)
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
# Plot the data
plt.plot(mean_id1, label='HIT')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)

id1 = pd.read_csv(f"results/{p}/gait/ID_results/Trial{trial}_generic_male_body.sto", sep='\t', skiprows=6)
gc_list = []

for gc in gc_filtered:
    start, end = gc
    # Get the index of the first and last gait cycle
    id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
    # interpolate to 100 points
    id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
    gc_list.append(id1_gc)
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
#plt.plot(mean_id1, label='SMPL-generic')
#plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)


id1 = pd.read_csv(f"results/{p}/gait/ID_results/Trial{trial}_ignore_physics.sto", sep='\t', skiprows=6)
gc_list = []

for gc in gc_filtered:
    start, end = gc
    # Get the index of the first and last gait cycle
    id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
    # interpolate to 100 points
    id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
    gc_list.append(id1_gc)
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
plt.plot(mean_id1, label='Generic')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)

gc_list = []
for i in range(15):
    try:
        path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/generic/ID/Trial{trial}_segment_{i}_id.sto"
        df = pd.read_csv(path, sep='\t', skiprows=10)
    except:
        continue
    idx_0 = i * 2000
    for gc in gc_filtered:
        start, end = gc
        if end > idx_0 + 2000 or start < idx_0:
            continue
        id1_gc = df[dim].iloc[start-idx_0:end-idx_0].to_numpy().copy()
    try:
        id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
        gc_list.append(id1_gc)
    except:
        continue
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
plt.plot(mean_id1, label='addB-ID')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)
plt.legend()

gc_list = []
for i in range(15):
    try:
        path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/HIT/ID/Trial{trial}_segment_{i}_id.sto"
        df = pd.read_csv(path, sep='\t', skiprows=10)
    except:
        continue
    idx_0 = i * 2000
    for gc in gc_filtered:
        start, end = gc
        if end > idx_0 + 2000 or start < idx_0:
            continue
        id1_gc = df[dim].iloc[start-idx_0:end-idx_0].to_numpy().copy()
    try:
        id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
        gc_list.append(id1_gc)
    except:
        continue
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
plt.plot(mean_id1, label='HIT+addB-ID')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)
plt.legend()

plt.xlabel('Gait cycle [%]')
plt.ylabel('Contralateral knee moment [Nm]')
plt.xlim(0, 99)

# save as pdf
plt.savefig(f'plots/{dim}.pdf', bbox_inches='tight')

In [ ]:
p = 'P04'
trial = 3
mass = 44.91

grf_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/data/{p}/gait/Trial{trial}_filtered.mot"
grf = pd.read_csv(grf_path, sep='\t', skiprows=5)

grf_y = grf['1_ground_force_vy'].to_numpy().copy()
grf_y[grf_y<30] = 0
grf_y[grf_y>0] = 1

# Find all rising edges
rising_edges = np.where(np.diff(grf_y) == 1)[0]
# Plot the data from the first rising edge to the next falling edge

grf_y = grf['1_ground_force_vy'].to_numpy()
gc_list = []    
for i in range(len(rising_edges)-1):
    start = rising_edges[i]
    end = rising_edges[i+1]
    gc_list.append((start, end))

# plot the lengths of each gaait cycle
plt.figure(figsize=(5,6))
mean_gc_len = np.mean([end-start for start, end in gc_list])
std_gc_len = np.std([end-start for start, end in gc_list])
gc_filtered = []
for start, end in gc_list:
    if abs(end-start-mean_gc_len) < std_gc_len:
        gc_filtered.append((start, end))

# loadID 
id1 = pd.read_csv(f"results/{p}/gait/ID_results/Trial{trial}_HIT_body.sto", sep='\t', skiprows=6)
id_1_ik = pd.read_csv(f"results/{p}/gait/addB-ignore_phys/IK/Trial{trial}_filtered.mot", sep='\t', skiprows=10)
dim = "knee_angle_r_moment"
dim_ik = "knee_angle_r"
# Grab all gait cycles
gc_list = []

for gc in gc_filtered:
    start, end = gc
    # Get the index of the first and last gait cycle
    id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
    id1_ik_gc = id_1_ik[dim_ik].iloc[start:end].to_numpy().copy()
    # interpolate to 100 points
    id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
    id1_ik_gc = np.interp(np.linspace(0, len(id1_ik_gc)-1, 100), np.arange(len(id1_ik_gc)), id1_ik_gc)
    gc_list.append(id1_gc*id1_ik_gc/100)
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
# Plot the data
plt.plot(mean_id1, label='HIT')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)

gc_list = []
id1 = pd.read_csv(f"results/{p}/gait/ID_results/Trial{trial}_generic_male_body.sto", sep='\t', skiprows=6)
for gc in gc_filtered:
    start, end = gc
    # Get the index of the first and last gait cycle
    id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
    id1_ik_gc = id_1_ik[dim_ik].iloc[start:end].to_numpy().copy()
    # interpolate to 100 points
    id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
    id1_ik_gc = np.interp(np.linspace(0, len(id1_ik_gc)-1, 100), np.arange(len(id1_ik_gc)), id1_ik_gc)
    gc_list.append(id1_gc*id1_ik_gc/100)
mean_id1 = np.mean(gc_list, axis=0)
stgd_id1 = np.std(gc_list, axis=0)
#plt.plot(mean_id1, label='SMPL-generic')
#plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)


gc_list = []
id1 = pd.read_csv(f"results/{p}/gait/ID_results/Trial{trial}_ignore_physics.sto", sep='\t', skiprows=6)
for gc in gc_filtered:
    start, end = gc
    # Get the index of the first and last gait cycle
    id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
    id1_ik_gc = id_1_ik[dim_ik].iloc[start:end].to_numpy().copy()
    # interpolate to 100 points
    id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
    id1_ik_gc = np.interp(np.linspace(0, len(id1_ik_gc)-1, 100), np.arange(len(id1_ik_gc)), id1_ik_gc)
    gc_list.append(id1_gc*id1_ik_gc/100)
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
plt.plot(mean_id1, label='Generic')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)

gc_list = []
for i in range(15):
    try:
        path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/generic/ID/Trial{trial}_segment_{i}_id.sto"
        df = pd.read_csv(path, sep='\t', skiprows=10)
        path2 = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/generic/IK/Trial{trial}_segment_{i}_id.mot"
        df2 = pd.read_csv(path2, sep='\t', skiprows=10)
    except:
        continue
    idx_0 = i * 2000
    for gc in gc_filtered:
        start, end = gc
        if end > idx_0 + 2000 or start < idx_0:
            continue
        id1_gc = df[dim].iloc[start-idx_0:end-idx_0].to_numpy().copy()
        id1_ik_gc = df2[dim_ik].iloc[start-idx_0:end-idx_0].to_numpy().copy()
    try:
        id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
        id1_ik_gc = np.interp(np.linspace(0, len(id1_ik_gc)-1, 100), np.arange(len(id1_ik_gc)), id1_ik_gc)
        gc_list.append(id1_gc*id1_ik_gc/100)
    except:
        continue
mean_id1 = np.mean(gc_list, axis=0)
std_id1 = np.std(gc_list, axis=0)
plt.plot(mean_id1, label='addB-ID')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)
plt.legend()

gc_list = []
for i in range(15):
    try:
        path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/results/{p}/gait/addB-with_phys/HIT/ID/Trial{trial}_segment_{i}_id.sto"
        df = pd.read_csv(path, sep='\t', skiprows=10)
    except:
        continue
    idx_0 = i * 2000
    for gc in gc_filtered:
        start, end = gc
        if end > idx_0 + 2000 or start < idx_0:
            continue
        id1_gc = df[dim].iloc[start-idx_0:end-idx_0].to_numpy().copy()
    try:
        id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
        gc_list.append(id1_gc)
    except:
        continue
#mean_id1 = np.mean(gc_list, axis=0)
#std_id1 = np.std(gc_list, axis=0)
#plt.plot(mean_id1, label='HIT+addB-ID')
#plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)
plt.legend()
#
plt.xlabel('Gait cycle [%]')
plt.ylabel(f'{dim_ik} power [W/kg]')
plt.xlim(0, 99)
#
# save as pdf
plt.savefig(f'plots/power_{dim}.pdf', bbox_inches='tight')
plt.grid()

In [ ]:
gc_list = []
id1 = pd.read_csv(f"results/{p}/gait/ID_results/Trial{trial}_generic_male_body.sto", sep='\t', skiprows=6)
for gc in gc_filtered:
    start, end = gc
    # Get the index of the first and last gait cycle
    id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
    id1_ik_gc = id_1_ik[dim_ik].iloc[start:end].to_numpy().copy()
    # interpolate to 100 points
    id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
    id1_ik_gc = np.interp(np.linspace(0, len(id1_ik_gc)-1, 100), np.arange(len(id1_ik_gc)), id1_ik_gc)
    gc_list.append(id1_gc*id1_ik_gc/100)
mean_id1 = np.mean(gc_list, axis=0)
stgd_id1 = np.std(gc_list, axis=0)
plt.plot(mean_id1, label='SMPL-generic')
plt.fill_between(np.arange(len(mean_id1)), mean_id1-std_id1, mean_id1+std_id1, alpha=0.2)

In [ ]:

end

In [ ]:
id_1_ik